# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and performing basic data analysis on the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from the schema
dataset = mlc.Dataset(croissant_url)

# Print out the name and description from metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs so we know what is available for extraction and analysis.

**All Croissant dataset entities must be referenced by their `@id` values.**

In [ ]:
# Explore available record sets, their fields and IDs
def print_record_sets(ds):
    print("Available record sets (@id):")
    for record_set in ds.record_sets:
        rec_id = record_set['@id']
        name = record_set.get('name', '(no name)')
        print(f"- Record Set: {rec_id} ({name})")
        if 'field' in record_set:
            fields = record_set['field']
            if isinstance(fields, dict):
                fields = [fields]
            print("    Fields:")
            for field in fields:
                fid = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
                fname = field.get('name', '') if isinstance(field, dict) else ''
                print(f"      - {fid} {f'({fname})' if fname else ''}")
        else:
            print("    [No fields listed]")
        print()

# Print the overview
print_record_sets(dataset)

## 3. Data Extraction
We'll extract all available record sets and load them as Pandas DataFrames. We'll use the `@id` for each record set and for the fields.

In [ ]:
# Extract all record set @ids from the dataset metadata

record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
print("Found record sets:")
print(record_sets_ids)

# Load records for each record set into a DataFrame and store in a dictionary
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  -> Loaded {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print("  -> No records found (empty record set or no data accessible)")

# If record sets are present, display summary of columns for the first one
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set {first_record_set_id}: {dataframes[first_record_set_id].columns.tolist()}")
    dataframes[first_record_set_id].head()
else:
    print("No dataframes could be created (no record sets populated with data available via mlcroissant).")

## 4. Exploratory Data Analysis (EDA)
You can now analyze the tabular data. For demonstration, we will:
- Select a record set and numeric field (by `@id`)
- Filter for values above a threshold
- Normalize the numeric field
- Group by another categorical field

Replace the variables below with the relevant `@id` values for your dataset.

In [ ]:
# Set up IDs for record set and fields for EDA. Update these as appropriate for this dataset.

# If there are no dataframes, this cell will not run
if dataframes:
    # Automatically select the first available record set for demo (customize as needed!)
    record_set_id = first_record_set_id
    df = dataframes[record_set_id]

    # Try to automatically detect first numeric field (float/int) by checking dtypes
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        # Fallback: use first column
        numeric_field_id = df.columns[0]

    print(f"Analyzing record set: {record_set_id}")
    print(f"Using numeric field: {numeric_field_id}")

    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    # Filter for rows above mean (or above 0 if not numeric)
    try:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
    except Exception as e:
        print(f"Could not filter on field {numeric_field_id}. Error: {e}")
        filtered_df = df.copy()

    # Normalize numeric field
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
    else:
        print(f"Cannot normalize non-numeric field {numeric_field_id}")

    # Try to detect group-by candidate (first non-numeric field)
    categorical_fields = [c for c in df.columns if df[c].dtype == 'object']
    if categorical_fields:
        group_field_id = categorical_fields[0]
        # Group by and compute mean for numeric columns
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No categorical/group fields found in columns.")
else:
    print("No dataframes available for EDA step.")

## 5. Visualization
Let's plot the distribution of the (automatically selected) numeric field, and if possible, visualize the means grouped by the selected group field. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Distribution of numeric field after filtering
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id} (filtered)")
        plt.xlabel(numeric_field_id)
        plt.show()

    # If grouped_df exists, make a bar plot of means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No data available to visualize.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to load metadata from the FAIR² dataset, enumerated all available record sets and their `@id`s, then extracted and explored the tabular contents of the data using their unique identifiers. We demonstrated standard exploratory and visualization techniques using dynamically selected fields. For detailed analysis, consult the record set and field `@id`s for more targeted queries and ensure fair treatment of sensitive columns (e.g., gender, socio-economic status) as called out in the dataset documentation.